In [ ]:
import ast
import os
import warnings

import pandas as pd

from animal_re_id.config import load_paths, resolve_path
from animal_re_id.datasets import TurtlesOfSMSRC
from animal_re_id.models import EncounterPredictions, merge_data
from animal_re_id.similarity import SimilarityCache, SimilarityRunSpec, get_similarity_dataset_config

warnings.filterwarnings("ignore")

In [ ]:
local_path = "../../config/paths.json"
paths = load_paths("TurtlesOfSMSRC", local_path=local_path)

B_max = 100
max_per_method = 80
max_per_identity = 2

root = resolve_path(paths["root"])
dataset_name = "TurtlesOfSMSRC"
path_replace_identities = "hidden/matched_encounters.txt"
device = "cuda"

config = get_similarity_dataset_config(dataset_name, B=B_max)
img_load = config.img_load
matching_parts = config.matching_parts
method_names = [spec.name for spec in config.pipeline_specs]

In [ ]:
root_results = resolve_path(paths["root_similarity"])
cache = SimilarityCache(root_results)

In [ ]:
with open(path_replace_identities) as f:
    replace_identity = ast.literal_eval(f.read())

os.makedirs(root_results, exist_ok=True)
dataset_kwargs = {"img_load": img_load, "replace_identity": replace_identity}
database_query_matching = False
dataset = TurtlesOfSMSRC(root, database_query_matching, dataset_kwargs=dataset_kwargs)

In [ ]:
results_all = {}
query_all = {}
database_all = {}
ignore_policy = config.ignore_pairs_policies[0]
splitter = config.splitters[0]
for matching_part in matching_parts:
    predictions_by_pipeline = []
    for pipeline_spec in config.pipeline_specs:
        run_spec = SimilarityRunSpec(pipeline_spec, ignore_policy, splitter, matching_part, img_load)
        saved_vars = run_spec.load(cache)
        encounter_predictions = EncounterPredictions(saved_vars["similarity"], saved_vars["query"], saved_vars["database"])
        encounter_predictions.group_by_encounter()
        encounter_predictions.classify(pipeline_spec.name, max_per_method)
        predictions_by_pipeline.append(encounter_predictions)

    merged = merge_data(predictions_by_pipeline)
    results_all[matching_part] = merged.extract_highest_predictions(max_per_method * len(method_names), max_per_identity)

In [ ]:
results_save = pd.concat(results_all.values())
if "key" in results_save.columns:
    results_save = results_save.drop(["key"], axis=1)
results_save.to_csv(f"../../verification_data_{dataset_name}.csv", index=False)

In [ ]:
from animal_re_id.models import plot_matches

dataset = TurtlesOfSMSRC(root, database_query_matching, dataset_kwargs=dataset_kwargs)
query_all = dataset.wildlife_dataset

n_plot = 5
for matching_part in matching_parts:
    for _, result in results_all[matching_part].iloc[:n_plot].iterrows():
        plot_matches(query_all, result)